# Get New Bidirectional regions to count for ATAC-seq

I am using 1kb regions and merging any bidirectionals overlapping to ensure proper normalization because the non-strandedness of ATAC-seq makes it almost impossible to deconvolute between overlapping enhancers.

In [16]:
library(data.table)
library(stringr)

In [5]:
wd = "~/projects/Resp_Env/"
mus <- fread(paste0(wd, "Comb_UPM_WSP_ADP/mumerge/out/UPM_WSP_ADP_tfit_MUMERGE.bed"))
mus[1:2,]

V1,V2,V3,V4
<chr>,<int>,<int>,<chr>
chr1,629230,629302,APM_sm1_1;APM_sm1_2;VEH_sm1_1;VEH_sm1_2
chr1,629882,629968,APM_sm1_1;APM_sm1_2;UPM120_sm36_1;UPM120_sm36_2;UPM30_sm36_1;UPM30_sm36_2;VEH_B2B_1;VEH_B2B_2;VEH_sm1_1;VEH_sm1_2;VEH_sm36_1;VEH_sm36_2;WSP120_B2B_1;WSP120_B2B_2;WSP30_B2B_1;WSP30_B2B_2


In [8]:
mus$mu <- as.integer((mus$V3 + mus$V2)/2)
mus$below <- mus$mu - 500
mus$above <- mus$mu + 500
mus$name <- paste0(mus$V1, ":", mus$mu)

mus[1:4,]

V1,V2,V3,V4,mu,below,above,name
<chr>,<int>,<int>,<chr>,<int>,<dbl>,<dbl>,<chr>
chr1,629230,629302,APM_sm1_1;APM_sm1_2;VEH_sm1_1;VEH_sm1_2,629266,628766,629766,chr1:629266
chr1,629882,629968,APM_sm1_1;APM_sm1_2;UPM120_sm36_1;UPM120_sm36_2;UPM30_sm36_1;UPM30_sm36_2;VEH_B2B_1;VEH_B2B_2;VEH_sm1_1;VEH_sm1_2;VEH_sm36_1;VEH_sm36_2;WSP120_B2B_1;WSP120_B2B_2;WSP30_B2B_1;WSP30_B2B_2,629925,629425,630425,chr1:629925
chr1,631327,631395,APM_sm1_1;VEH_sm1_1;VEH_sm1_2,631361,630861,631861,chr1:631361
chr1,632395,632457,VEH_sm1_2,632426,631926,632926,chr1:632426


In [10]:
# get the merged version of this to ensure no double counting

write.table(mus[,c("V1", "below", "above", "name")], "./UPM_WSP_ADP_tfit_MUMERGE_1kb.bed", quote=FALSE, sep="\t", row.names=FALSE, col.names=FALSE)

In [ ]:
bedtools merge -i ./UPM_WSP_ADP_tfit_MUMERGE_1kb.bed -c 4 -o collapse > ./UPM_WSP_ADP_tfit_MUMERGE_1kb_merged.bed
# checked and all were sorted before

In [12]:
# read in merged
merged <- fread("./UPM_WSP_ADP_tfit_MUMERGE_1kb_merged.bed")
dim(merged)
merged[1:2,]
dim(mus)

[1] 68613     4

V1,V2,V3,V4
<chr>,<int>,<int>,<chr>
chr1,628766,630425,"chr1:629266,chr1:629925"
chr1,630861,631861,chr1:631361


[1] 85021     8

In [17]:
# save as  feature counts SAF files
# GeneID , Chr , Start , End and Strand
# the dedup use chromosome 1,2,3 not chr1 so use that

mu_saf <- as.data.frame(data.table("GeneID"=mus$name, "Chr"=str_split_fixed(mus$V1, "chr",2)[,2], "Start"=mus$below, "End"=mus$above, "Strand"="."))
merged_saf <- as.data.frame(data.table("GeneID"=merged$V4, "Chr"=str_split_fixed(merged$V1, "chr",2)[,2], "Start"=merged$V2, "End"=merged$V3, "Strand"="."))
mu_saf[1:2,]
merged_saf[1:2,]

write.table(mu_saf, "UPM_WSP_ADP_tfit_MUMERGE_1kb.saf", quote=FALSE, sep="\t", row.names=FALSE)
write.table(merged_saf, "UPM_WSP_ADP_tfit_MUMERGE_1kb_merged.saf", quote=FALSE, sep="\t", row.names=FALSE)

,GeneID,Chr,Start,End,Strand
,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,chr1:629266,1,628766,629766,.
2,chr1:629925,1,629425,630425,.


,GeneID,Chr,Start,End,Strand
,<chr>,<chr>,<int>,<int>,<chr>
1,"chr1:629266,chr1:629925",1,628766,630425,.
2,chr1:631361,1,630861,631861,.
